# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [1]:
"""The following generates the Quiz all our models will be evaluated on."""

import itertools
import string
import numpy as np

import logging

from ordered_set import OrderedSet
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
load_dotenv("./keys.env", verbose=True)

from smolbench.induction.chromatic import (
    ChromaticIntervalsConfig,
    Prompter,
    get_random_exclusive_quiz,
    anneal_intervals,
)
from smolbench.evals import Marks
from smolbench.evals.openrouter import evaluate

template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. The following lists the people who were $role and"
    " the years they were $role:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "Between the years $start and $end, exclusive of the end, could $color"
    " have headed the $parade parade every year?"
)


def query_gen(
    labels_to_intervals: Dict[Color, Intervals],
    interval_to_label: Dict[Intervals, Color],
    seed: int,
) -> Dict[str, str]:
    """Generates a series of queries"""
    rng: np.random.Generator = np.random.default_rng(seed)
    # Finds max interval.
    n: int = max(interval[1] for interval in interval_to_label)
    for color, intervals in labels_to_intervals.items():
        # Generates a series of true items.
        for start, end in intervals:
            start, end = np.sort(
                rng.choice(range(start, end + 1), size=2, replace=False)
            )
            yield {"color": color, "start": start, "end": end}, True
        # Generates a false proposition.
        invalid_range: Intervals = anneal_intervals(
            itertools.chain(
                (
                    OrderedSet(interval_to_label.keys())
                    - OrderedSet(itertools.chain(*intervals))
                )
            )
        )
        for start, end in invalid_range:
            start = rng.choice(range(start, end))
            # Binom with p = intervals / n capped at end for a similar-ish
            # distr. to positive accounts.
            end = min(
                end,
                start
                + rng.binomial(
                    end - start + 1,
                    np.mean([len(interval) for interval in interval_to_label]) / n,
                )
                + 1,
            )
            yield {"color": color, "start": start, "end": end}, False


SEED: int = 1776
intens_quiz, extens_quiz = get_random_exclusive_quiz(
    ChromaticIntervalsConfig(
        n=int(52 * 250),
        intervals= 250 // 4,
        colors=45,
        seed=SEED,
    ),
    Prompter(
        template,
        {
            "role": "Twislax",
            "parade": "Gildane",
        },
        query_gen,
    ),
)

## Prompt Validation

In [2]:
print(intens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists the people who were Twislax and the years they were Twislax:
qo was Twislax on 1649 to 1978.
Lw was Twislax on 5775 to 6110 and 7578 to 7769.
Gv was Twislax on 1979 to 2336 and 7042 to 7247.
Wd was Twislax on 3655 to 4101.
UE was Twislax on 9712 to 10295.
ld was Twislax on 11710 to 11944.
Gt was Twislax on 4982 to 5473.
nR was Twislax on 2593 to 2794 and 10371 to 10416.
Ef was Twislax on 0 to 796, 9376 to 9601, and 11113 to 11479.
wJ was Twislax on 3559 to 

In [3]:
print(extens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists the people who were Twislax and the years they were Twislax:
qo was Twislax on 1649, 1650, 1651, 1652, 1653, 1654, 1655, 1656, 1657, 1658, 1659, 1660, 1661, 1662, 1663, 1664, 1665, 1666, 1667, 1668, 1669, 1670, 1671, 1672, 1673, 1674, 1675, 1676, 1677, 1678, 1679, 1680, 1681, 1682, 1683, 1684, 1685, 1686, 1687, 1688, 1689, 1690, 1691, 1692, 1693, 1694, 1695, 1696, 1697, 1698, 1699, 1700, 1701, 1702, 1703, 1704, 1705, 1706, 1707, 1708, 1709, 1710, 1711, 1712

# Decoder-Only Model
This section tests classical decoder-only models.

In [4]:
decode_intens_eval: Marks = evaluate(
    intens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED
)

INFO:root:Response:
{'id': 'gen-1775118174-6Xb69pKAGk1jdy2SJ6du', 'object': 'chat.completion', 'created': 1775118174, 'model': 'mistralai/mistral-7b-instruct-v0.1', 'provider': 'Cloudflare', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': ' False', 'refusal': None, 'reasoning': None}}], 'usage': {'prompt_tokens': 1225, 'completion_tokens': 2, 'total_tokens': 1227, 'cost': 0.00013513, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00013513, 'upstream_inference_prompt_cost': 0.00013475, 'upstream_inference_completions_cost': 3.8e-07}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}}
 was 1227 <= 2824
INFO:root:Response:
{'id': 'gen-1775118174-Q6YaTDcp2ryEv0e0yErN', 'object': 'chat.completion', 'cre

In [5]:
decode_extens_eval: Marks = evaluate(
    extens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED
)

INFO:root:Response:
{'id': 'gen-1775118363-wuQLnugKPB1XmxDbLN8a', 'object': 'chat.completion', 'created': 1775118363, 'model': 'mistralai/mistral-7b-instruct-v0.1', 'provider': 'Cloudflare', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': ' False', 'refusal': None, 'reasoning': None}}], 'usage': {'prompt_tokens': 1463, 'completion_tokens': 2, 'total_tokens': 1465, 'cost': 0.00016131, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00016131, 'upstream_inference_prompt_cost': 0.00016093, 'upstream_inference_completions_cost': 3.8e-07}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}}
 was 1465 <= 2824
INFO:root:Response:
{'id': 'gen-1775118363-xQrFYiDx8z9lvceCi8pn', 'object': 'chat.completion', 'cre

In [4]:
print(
    decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid
)
print(
    decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid
)

38 56 13
25 41 41


## CoT Model
This section tests a CoT model.

In [6]:
cot_intens_eval: Marks = evaluate(
    intens_quiz, "qwen/qwen3-14b", SEED, extra_args={"reasoning": {"enabled": True}}
)

INFO:root:Response:
{'id': 'gen-1775118381-bfYHo10hIKAYrC9AlecR', 'object': 'chat.completion', 'created': 1775118381, 'model': 'qwen/qwen3-14b-04-28', 'provider': 'DeepInfra', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': 'False', 'refusal': None, 'reasoning': "\nOkay, let's tackle this question. The user is asking if Lw could have headed the Gildane parade every year between 5738 and 5740, not including 5740. First, I need to check Lw's tenure as Twislax.\n\nFrom the context, Lw was Twislax on two periods: 5775 to 6110 and 7578 to 7769. The question is about the years 5738 to 5740. Let me note that 5738 is before 5775, so Lw's first period starts way after the end of the questioned range.\n\nWait, the end of the range is exclusive, so up to 5739.999... So the years in question are 5738 and 5739. But Lw's tenure starts at 5775. That's way after 5739. Therefore, 

In [7]:
cot_extens_eval: Marks = evaluate(
    extens_quiz, "qwen/qwen3-14b", SEED, extra_args={"reasoning": {"enabled": True}}
)

ValueError: Response:
{'id': 'gen-1775118819-EKpbS1k7FJPgQwTUTd1P', 'object': 'chat.completion', 'created': 1775118819, 'model': 'qwen/qwen3-14b-04-28', 'provider': 'Alibaba', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': 'False', 'refusal': None, 'reasoning': None}}], 'usage': {'prompt_tokens': 80324, 'completion_tokens': 5, 'total_tokens': 80329, 'cost': 0.01827826, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.0281204, 'upstream_inference_prompt_cost': 0.0281134, 'upstream_inference_completions_cost': 7e-06}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}}
 was 80329 > 40960

In [ ]:
print(cot_intens_eval.correct, cot_intens_eval.incorrect, cot_intens_eval.invalid)
print(cot_extens_eval.correct, cot_extens_eval.incorrect, cot_extens_eval.invalid)

107 0 0
104 3 0


## MoE Model
This section tests an MoE model.

In [ ]:
moe_intens_eval: Marks = evaluate(intens_quiz, "openai/gpt-oss-20b", SEED)

In [ ]:
moe_extens_eval: Marks = evaluate(extens_quiz, "openai/gpt-oss-20b", SEED)

In [12]:
print(moe_intens_eval.correct, moe_intens_eval.incorrect, moe_intens_eval.invalid)
print(moe_extens_eval.correct, moe_extens_eval.incorrect, moe_extens_eval.invalid)

106 1 0
100 7 0


# HRM Model
This section tests an HRM model.

In [ ]:
hrm_intens_eval: Marks = evaluate(intens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
hrm_extens_eval: Marks = evaluate(extens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
print(hrm_intens_eval.correct, hrm_intens_eval.incorrect, hrm_intens_eval.invalid)
print(hrm_extens_eval.correct, hrm_extens_eval.incorrect, hrm_extens_eval.invalid)